##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Live API: Thinking and background reasoning

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_Live_Thinking.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini Live API enables real-time, bidirectional audio and video conversations with Gemini models.

Standard voice models work well for immediate back-and-forth dialogue. But when a request requires planning, multi-step analysis, or external tools, direct responses hit a limit: the model must either answer without deep reasoning or pause silently while waiting for tools to finish.

**Gemini 3.8 Live Extended Thinking** (`gemini-3.8-live-extended-thinking`) adds background reasoning to real-time voice sessions. The model plans and executes asynchronous tools in the background while streaming natural conversational fillers to keep the interaction active and engaging.

This architecture introduces two key conversational lifecycle patterns:

- **Conversational fillers**: The model speaks intermediate updates (such as *"Checking flight options now"*) while executing tools in the background.
- **Interaction status tracking**: Because the model can speak multiple times during a single request, the server emits `interaction_status: "IN_PROGRESS"` during background processing and `interaction_status: "IDLE"` when the overall task completes.

In this guide, you will learn how to:
1. Configure background reasoning using `thinking_config`.
2. Declare asynchronous non-blocking tools (`behavior: "NON_BLOCKING"`).
3. Track session lifecycles using `interaction_status` instead of relying solely on `turn_complete`.
4. Handle intermediate audio fillers and asynchronous tool responses.

## Setup

### Install SDK

In [1]:
%pip install -U -q "google-genai>=2.9.0"

Note: you may need to restart the kernel to use updated packages.


### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you are not sure how to create a Colab Secret, see the [Authentication](../quickstarts/Authentication.ipynb) quickstart for a walkthrough.

In [2]:
import os
from google import genai

try:
  from google.colab import userdata

  GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
  GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
client = genai.Client(api_key=GEMINI_API_KEY)

### Select a model

Select the Live API model you want to use. `gemini-3.8-live-extended-thinking` is optimized for multi-step reasoning and background tool execution, while `gemini-3.8-live` delivers lowest latency for direct voice interactions.

In [3]:
MODEL_ID = "gemini-3.8-live-extended-thinking"  # @param ["gemini-3.8-live", "gemini-3.8-live-extended-thinking", "gemini-3.1-flash-live-preview"] {"allow-input": true, "isTemplate": true}
THINKING_LEVEL = "LOW"  # @param ["LOW", "HIGH"] {"allow-input": true, "isTemplate": true}

### Audio playback helper

The Live API returns raw 24kHz PCM audio frames. Use a standard context manager to write the stream to a `.wav` file for playback.

In [4]:
import contextlib
import wave


@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
  """Context manager to write raw PCM audio frames to a WAV file."""
  with wave.open(filename, "wb") as wf:
    wf.setnchannels(channels)
    wf.setsampwidth(sample_width)
    wf.setframerate(rate)
    yield wf

## Configure Thinking and non-blocking tools

When using `gemini-3.8-live-extended-thinking`, tool declarations must specify `behavior="NON_BLOCKING"`. Synchronous blocking tools will return an error.

You can configure reasoning depth using `thinking_config=types.ThinkingConfig(thinking_level=...)`, supporting `"low"`, `"medium"`, or `"high"`.

In [5]:
from google.genai import types

# Declare an asynchronous non-blocking function
search_flights = types.FunctionDeclaration(
    name="search_flights",
    description="Searches for available flights between cities.",
    behavior="NON_BLOCKING",
    parameters={
        "type": "OBJECT",
        "properties": {
            "origin": {"type": "STRING"},
            "destination": {"type": "STRING"},
        },
        "required": ["origin", "destination"],
    },
)

# Session configuration with Thinking and non-blocking tools
config = types.LiveConnectConfig(
    response_modalities=["AUDIO"],
    thinking_config=types.ThinkingConfig(thinking_level=THINKING_LEVEL),
    tools=[types.Tool(function_declarations=[search_flights])],
)

## Run a Live session with background reasoning

In a Thinking session, the model can emit intermediate conversational fillers (such as *"Searching flights now"*) while reasoning and awaiting tool responses.

Inspect `message.interaction_status` on incoming server messages:
- `"IN_PROGRESS"`: The model is actively reasoning, streaming conversational fillers, or awaiting tool responses.
- `"IDLE"`: The model has completed reasoning, executed all tools, and delivered its final answer. The session is now idle and ready for user input.

In [6]:
import asyncio
from IPython.display import Audio, display


async def simulate_flight_search(origin: str, destination: str) -> dict:
  """Simulates an external flight lookup API call."""
  print(f"\n[Tool execution] Searching flights from {origin} to {destination}...")
  await asyncio.sleep(2)
  return {
      "flights": [
          {"flight": "PA 204", "departure": "10:30 AM", "price": "$189"},
          {"flight": "GA 512", "departure": "02:15 PM", "price": "$220"},
      ]
  }


async def run_thinking_session():
  prompt = "Find me a flight from New York to London tomorrow, and tell me the cheapest option."
  print(f"> User: {prompt}\n")

  file_name = "thinking_live_output.wav"
  with wave_file(file_name) as wav:
    async with client.aio.live.connect(model=MODEL_ID, config=config) as session:
      await session.send_realtime_input(text=prompt)

      async for message in session.receive():
        # Track interaction status for server lifecycle
        status = getattr(message, "interaction_status", None)
        if status:
          print(f"\n[Interaction status]: {status}")

        # Process audio chunks (conversational fillers or final speech)
        if message.data is not None:
          wav.writeframes(message.data)
          print(".", end="")

        if message.server_content and message.server_content.model_turn:
          for part in message.server_content.model_turn.parts:
            if part.inline_data:
              wav.writeframes(part.inline_data.data)
              print(".", end="")

        # Handle asynchronous non-blocking tool call
        if message.tool_call:
          for call in message.tool_call.function_calls:
            print(f"\n[Server requested tool]: {call.name} with {call.args}")
            tool_output = await simulate_flight_search(**call.args)
            response = types.FunctionResponse(
                id=call.id,
                name=call.name,
                response={"result": tool_output},
            )
            print("[Sending tool response to Gemini Live]")
            await session.send_tool_response(function_responses=[response])

        # Only transition UI to idle when interaction_status is IDLE
        if status == "IDLE":
          print("\n[Turn complete: Session is idle and ready for user input]")
          break

  display(Audio(file_name, autoplay=True))


await run_thinking_session()

> User: Find me a flight from New York to London tomorrow, and tell me the cheapest option.

[Interaction status]: LISTENING
[Interaction status]: THINKING
................
[Server requested tool]: search_flights with {'destination': 'London', 'origin': 'New York'}

[Tool execution] Searching flights from New York to London...
[Sending tool response to Gemini Live]
[Interaction status]: RESPONDING
................................................................
[Interaction status]: IDLE
[Turn complete: Session is idle and ready for user input]


## Understanding turn completion vs interaction status

The key difference between standard Live API models and Thinking models is how conversational turns are managed:

| Model | Turn signal | Idle signal | Tool behavior |
| :--- | :--- | :--- | :--- |
| **Gemini 3.8 Live** (`gemini-3.8-live`) | `turn_complete == True` | `turn_complete == True` | Supports `BLOCKING` and `NON_BLOCKING` |
| **Gemini 3.8 Live Extended Thinking** (`gemini-3.8-live-extended-thinking`) | `turn_complete == True` finishes individual utterance | `interaction_status == "IDLE"` | Requires `NON_BLOCKING` tools |

When building client applications (such as WebRTC or mobile voice agents):
- **Do not close microphone or show idle UI on `turn_complete`** if `interaction_status == "IN_PROGRESS"`.
- **Keep listening for tool calls and further speech** until `interaction_status == "IDLE"` arrives.

## What's next

- Learn more about thinking architecture in the [Thinking in the Live API guide](https://ai.google.dev/gemini-api/docs/live-api/thinking).
- Explore low-latency voice interactions with [Gemini 3.8 Live](./Get_started_LiveAPI.ipynb).
- Explore function calling capabilities in [Get started with Live API tools](./Get_started_LiveAPI_tools.ipynb).
- Review the [Gemini 3.8 Live model card](https://ai.google.dev/gemini-api/docs/models/gemini-3.8-live) and [Gemini 3.8 Live Extended Thinking model card](https://ai.google.dev/gemini-api/docs/models/gemini-3.8-live-extended-thinking).
- Try the Live API in [Google AI Studio](https://aistudio.google.com/live?model=gemini-3.8-live-extended-thinking).